# 樹木医の手引き 4版 — OCRノートブック（Colab / YomiToku）

スキャンPDF(1.07GB・702ページ)を、日本語特化OCR **YomiToku** でMarkdownに起こします。
RAG（検索拡張生成）の知識ベースの元データを作るのが目的です。

進め方は **①準備 → ②ドライブ接続 → ③パイロット(数ページで精度確認) → ④本番バッチ → ⑤結合** の順。
上から順にセルを実行してください。

- 出力はGoogleドライブ（非公開）に保存します。**公開リポジトリには置かないでください**（書籍本文＝第三者の著作物）。
- YomiTokuは非商用ライセンス(CC BY-NC-SA)。将来これを業務ツールに組み込む段階で商用ライセンスを確認してください。

_© 2026 Koh Kitsukawa. (このノートブックのコードについて。OCR出力の書籍本文は日本緑化センターに著作権があります)_

## ステップ0：GPUを有効にする（重要）

Colabメニュー **「ランタイム」→「ランタイムのタイプを変更」→ ハードウェアアクセラレータ = T4 GPU** を選んで保存。
GPUなしだと通常モデルは非常に遅くなります。

## ステップ1：ライブラリのインストール

`yomitoku`（OCR本体）と `pypdf`（PDFのページ操作）を入れます。数分かかることがあります。

In [ ]:
!pip -q install yomitoku pypdf

import torch
print("GPU使用可否:", torch.cuda.is_available())
print("GPU名:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし（CPUになります）")

## ステップ2：Googleドライブに接続してPDFの場所を指定

元PDFはあらかじめGoogleドライブのMyDrive直下などに置いておいてください。
下のセルの **PDF_PATH と OUT_DIR だけ** 自分の環境に合わせて書き換えます。

**2冊目（緑化樹木腐朽病害ハンドブック）を処理するとき** も、変えるのはこの2行だけです。

> **ここでエラーになる場合の対処**
> - このセルはPDFの中身を読みません。エラーの原因はほぼ次の2つです。
>   1. **ドライブが二重マウント**（同じランタイムで再実行したとき）。→ 下のセルは自動で判定してスキップします。
>   2. **PDF_PATH のファイル名が実物と一致しない**（版数・全角/半角スペース・拡張子の違い、サブフォルダに置いてある等）。→ 見つからなければ **ドライブ内のPDF一覧を表示** するので、そこから正確なパスをコピーして PDF_PATH に貼り直してください。


In [ ]:
from google.colab import drive
import os, glob

# --- ドライブ接続（再実行しても安全に）---
# 同じランタイムで2回mountすると
#   ValueError: Mountpoint must not already contain files
# というエラーになるため、マウント済みかどうかを先に判定する。
if os.path.ismount('/content/drive'):
    print("ドライブは既にマウント済みです（再マウントしません）")
else:
    drive.mount('/content/drive')

# ↓↓↓ ここだけ自分の環境に合わせて変更 ↓↓↓
# 【1冊目】樹木医の手引き
# PDF_PATH = "/content/drive/MyDrive/樹木医の手引き4版.pdf"
# OUT_DIR  = "/content/drive/MyDrive/ocr_tebiki"
# 【2冊目】緑化樹木腐朽病害ハンドブック（← ファイル名は実物に合わせて）
PDF_PATH = "/content/drive/MyDrive/緑化樹木腐朽病害ハンドブック.pdf"
OUT_DIR  = "/content/drive/MyDrive/ocr_fukyu"
# ↑↑↑ 変更ここまで ↑↑↑

os.makedirs(OUT_DIR, exist_ok=True)

if os.path.exists(PDF_PATH):
    size_gb = os.path.getsize(PDF_PATH) / 1e9
    print(f"OK: PDFが見つかりました（{size_gb:.2f} GB）→ {PDF_PATH}")
    print(f"    出力先: {OUT_DIR}")
else:
    print("×  指定したPDFが見つかりません:")
    print("   ", repr(PDF_PATH))
    print("   ↓ ドライブ内のPDF一覧です。ここから正確なファイル名をコピーして PDF_PATH に貼り直してください。")
    print("   （版数・全角/半角スペース・拡張子の大文字小文字の違いに注意）")
    top = sorted(glob.glob('/content/drive/MyDrive/*.pdf'))
    if top:
        print("   ── MyDrive直下のPDF ──")
        for p in top:
            print("   -", repr(p))
    else:
        print("   MyDrive直下にPDFはありません。サブフォルダを探します…")
        sub = sorted(glob.glob('/content/drive/MyDrive/**/*.pdf', recursive=True))
        if sub:
            for p in sub[:50]:
                print("   -", repr(p))
        else:
            print("   MyDrive内にPDFが1つも見つかりませんでした。")
            print("   → PDFのアップロード先と、マウント先(/content/drive/MyDrive)を確認してください。")
    raise FileNotFoundError(
        "PDF_PATH が実在しません。上の一覧から正しいパスを PDF_PATH に設定して、このセルを再実行してください。"
    )


## ステップ3：パイロット（まず数ページで精度確認）

いきなり702ページを回さず、数ページだけOCRして読み取り精度を確かめます。
既定は **PDFの489〜492ページ**（本文が詰まっており、化学式 H₂S など難所も含むため、精度チェックに向く）。

- ここでいうページ番号は **PDFのページ番号（全702）** です。本の印刷ページ番号とはズレます。
- 表の読み取りを試したい場合は、下の PILOT_START / PILOT_END を表のあるページに変えてください。

In [ ]:
from pypdf import PdfReader, PdfWriter

# --- パイロット対象（PDFのページ番号・1始まり）---
PILOT_START = 489
PILOT_END   = 492

reader = PdfReader(PDF_PATH)
total = len(reader.pages)
print("総ページ数:", total)

# 既存テキスト層の確認（スキャンのみならほぼ空。文字が出れば既にOCR済みの可能性）
sample = reader.pages[PILOT_START-1].extract_text() or ""
print(f"[テキスト層] {PILOT_START}ページの文字数:", len(sample.strip()))
if len(sample.strip()) > 20:
    print("→ 既にテキストが入っている可能性。中身の先頭:", sample[:150])
else:
    print("→ テキスト層はほぼ無し。画像スキャンとしてOCRします。")

# パイロットPDFを書き出し
writer = PdfWriter()
for i in range(PILOT_START-1, min(PILOT_END, total)):
    writer.add_page(reader.pages[i])
with open("/content/pilot.pdf", "wb") as f:
    writer.write(f)
print("pilot.pdf 作成:", min(PILOT_END, total)-PILOT_START+1, "ページ")

### パイロットOCRの実行

通常モデル（高精度）で実行します。オプションの意味:
- `--ignore_line_break`：本の段組みによる行内改行を無視し、**段落を連結**（RAG向きの自然な文になる）
- `--ignore_ruby`：ふりがな（ルビ）を除去してノイズ削減
- `--combine`：複数ページの結果を1つのMarkdownに統合
- `-v`：検出・読み順を可視化した画像も出力（目視確認用）

In [ ]:
!yomitoku /content/pilot.pdf -f md -o /content/pilot_out -v --combine --ignore_line_break --ignore_ruby -d cuda

import glob
mds = sorted(glob.glob("/content/pilot_out/*.md"))
print("出力Markdown:", mds)
if mds:
    print("="*50)
    print(open(mds[0], encoding="utf-8").read()[:2500])

### パイロットの確認ポイント

出力Markdownと、`/content/pilot_out` 内の可視化画像(jpg)を見て、次を確認してください。
- 樹種名・菌名・数値・**化学式（H₂S などの下付き）**・専門用語が正しく読めているか
- 可視化画像で **読み順** が乱れていないか（段組みの取り違えがないか）

問題なければステップ4（本番）へ。精度が低い場合は、解像度不足の可能性があるので中止して相談してください。

## ステップ4：本番バッチOCR（全702ページ・チェックポイント方式）

全ページを **50ページ刻み** でOCRし、各バッチのMarkdownをドライブに保存します。
**チェックポイント方式**（＝処理済みのバッチはスキップ）なので、Colabが途中で切断されても、
このセルを再実行すれば続きから再開できます。所要時間の目安は全体で概ね1〜2時間程度です。

In [ ]:
import os, glob, subprocess
from pypdf import PdfReader, PdfWriter

BATCH = 50  # 1バッチのページ数
reader = PdfReader(PDF_PATH)
total = len(reader.pages)
batch_dir = os.path.join(OUT_DIR, "batches")
os.makedirs(batch_dir, exist_ok=True)

for start in range(0, total, BATCH):
    end = min(start+BATCH, total)
    tag = f"{start+1:04d}-{end:04d}"                 # 例: 0001-0050
    done_flag = os.path.join(batch_dir, f"{tag}.md")  # これがあれば処理済み
    if os.path.exists(done_flag):
        print(f"[skip] {tag} は処理済み")
        continue
    print(f"[OCR] {tag} を処理中 ...")

    # このバッチ分だけのPDFを作成
    writer = PdfWriter()
    for i in range(start, end):
        writer.add_page(reader.pages[i])
    part_pdf = f"/content/part_{tag}.pdf"
    with open(part_pdf, "wb") as f:
        writer.write(f)

    # OCR実行（subprocess = 別プロセスでyomitokuコマンドを実行）
    out_tmp = f"/content/out_{tag}"
    subprocess.run(
        ["yomitoku", part_pdf, "-f", "md", "-o", out_tmp,
         "--combine", "--ignore_line_break", "--ignore_ruby", "-d", "cuda"],
        check=True,
    )

    # 生成Markdownをドライブへ保存（チェックポイント）
    parts = sorted(glob.glob(os.path.join(out_tmp, "*.md")))
    text = "\n\n".join(open(p, encoding="utf-8").read() for p in parts)
    with open(done_flag, "w", encoding="utf-8") as f:
        f.write(text)
    os.remove(part_pdf)  # 一時PDFを掃除
    print(f"[done] {tag} 保存: {done_flag}")

print("すべてのバッチが完了しました。")

## ステップ5：全バッチを1つのMarkdownに結合

各バッチ間に `<!-- batch xxxx-xxxx -->` の印を入れて結合します。
この印は、あとで各チャンクをPDFのページ範囲に対応づける（メタデータ付与）ときに使えます。

In [ ]:
import glob, os

batch_dir = os.path.join(OUT_DIR, "batches")
files = sorted(glob.glob(os.path.join(batch_dir, "*.md")))
merged = os.path.join(OUT_DIR, "tebiki_full.md")

with open(merged, "w", encoding="utf-8") as out:
    for p in files:
        tag = os.path.splitext(os.path.basename(p))[0]
        out.write(f"\n\n<!-- batch {tag} -->\n\n")   # ページ対応づけ用の印
        out.write(open(p, encoding="utf-8").read())

print("結合完了:", merged)
print("バッチ数:", len(files))
print("文字数:", os.path.getsize(merged), "バイト")

## 次のステップ

- できた `tebiki_full.md` が、RAGの `build-embeddings` の入力になります。
- **2冊目（緑化樹木腐朽病害ハンドブック）** も、ステップ2の `PDF_PATH` と `OUT_DIR` を差し替えれば同じ手順で処理できます。
- 出力（batches/ と full.md）は **ドライブ内の非公開のまま** にしてください。公開リポジトリには置きません。

パイロットのMarkdownが出たら、その精度を共有してください。問題なければ本番→メタデータ設計（タグ辞書・章節対応）へ進みます。